In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.stats import kendalltau, spearmanr, multivariate_normal, norm, chi2, kruskal, ranksums
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
from factor_analyzer import FactorAnalyzer
from sklearn.decomposition import PCA
from itertools import combinations

In [2]:
# ---------------------------------------------------------
# 1. Load the Excel Workbook
# ---------------------------------------------------------
excel_path = "/home/david/Git/SDLT_Paper/Data/adiya_isla_scores.xlsx"
xls = pd.ExcelFile(excel_path)
sheet_name = "Whole"

if sheet_name not in xls.sheet_names:
        print(f"Warning: Sheet '{sheet_name}' not found in workbook. Skipping.")


df_raw = pd.read_excel(xls, sheet_name=sheet_name)

In [3]:
def compute_ordinal_correlations(
    df: pd.DataFrame, n_bootstraps: int = 2000, alpha: float = 0.05
) -> dict:
    """Computes Kendall's tau-b and Spearman's rho correlation matrices, exact p-values,

    and bias-corrected/percentile bootstrap confidence intervals for ordinal
    data.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame of shape (N, P) containing ordinal ratings (e.g., 27x9 on 0-2 scale).
    n_bootstraps : int
        Number of bootstrap resamples for confidence interval estimation (default: 2000).
    alpha : float
        Significance level for confidence intervals (default: 0.05 for 95% CIs).

    Returns:
    --------
    dict:
        Dictionary containing:
        - 'kendall_tau': DataFrame of Kendall's tau-b coefficients
        - 'kendall_pvalues': DataFrame of two-tailed p-values for tau-b
        - 'kendall_ci_lower': DataFrame of lower CI bounds for tau-b
        - 'kendall_ci_upper': DataFrame of upper CI bounds for tau-b
        - 'spearman_rho': DataFrame of Spearman's rho coefficients
        - 'spearman_pvalues': DataFrame of two-tailed p-values for rho
        - 'spearman_ci_lower': DataFrame of lower CI bounds for rho
        - 'spearman_ci_upper': DataFrame of upper CI bounds for rho
    """
    cols = df.columns
    n_cols = len(cols)
    n_rows = len(df)

    # Initialize result containers
    tau_matrix = pd.DataFrame(np.eye(n_cols), index=cols, columns=cols)
    tau_pvals = pd.DataFrame(np.zeros((n_cols, n_cols)), index=cols, columns=cols)
    tau_ci_low = pd.DataFrame(np.eye(n_cols), index=cols, columns=cols)
    tau_ci_high = pd.DataFrame(np.eye(n_cols), index=cols, columns=cols)

    rho_matrix = pd.DataFrame(np.eye(n_cols), index=cols, columns=cols)
    rho_pvals = pd.DataFrame(np.zeros((n_cols, n_cols)), index=cols, columns=cols)
    rho_ci_low = pd.DataFrame(np.eye(n_cols), index=cols, columns=cols)
    rho_ci_high = pd.DataFrame(np.eye(n_cols), index=cols, columns=cols)

    np.random.seed(42)  # For reproducible bootstrap intervals

    for i in range(n_cols):
        for j in range(i + 1, n_cols):
            col_a = df[cols[i]].values
            col_b = df[cols[j]].values

            # --- 1. Point Estimates and Exact/Asymptotic p-values ---
            tau_res = kendalltau(col_a, col_b, variant="b")
            rho_res = spearmanr(col_a, col_b)

            tau_matrix.iloc[i, j] = tau_matrix.iloc[j, i] = tau_res.statistic
            tau_pvals.iloc[i, j] = tau_pvals.iloc[j, i] = tau_res.pvalue

            rho_matrix.iloc[i, j] = rho_matrix.iloc[j, i] = rho_res.statistic
            rho_pvals.iloc[i, j] = rho_pvals.iloc[j, i] = rho_res.pvalue

            # --- 2. Bootstrap Confidence Intervals for Small N (N=27) ---
            boot_taus = []
            boot_rhos = []

            for _ in range(n_bootstraps):
                boot_idx = np.random.choice(n_rows, size=n_rows, replace=True)
                boot_a = col_a[boot_idx]
                boot_b = col_b[boot_idx]

                # Compute bootstrap sample statistics
                b_tau = kendalltau(boot_a, boot_b, variant="b").statistic
                b_rho = spearmanr(boot_a, boot_b).statistic

                if not np.isnan(b_tau):
                    boot_taus.append(b_tau)
                if not np.isnan(b_rho):
                    boot_rhos.append(b_rho)

            # Percentile bootstrap bounds
            low_pct = (alpha / 2) * 100
            high_pct = (1 - alpha / 2) * 100

            tau_ci_low.iloc[i, j] = tau_ci_low.iloc[j, i] = (
                np.percentile(boot_taus, low_pct) if boot_taus else np.nan
            )
            tau_ci_high.iloc[i, j] = tau_ci_high.iloc[j, i] = (
                np.percentile(boot_taus, high_pct) if boot_taus else np.nan
            )

            rho_ci_low.iloc[i, j] = rho_ci_low.iloc[j, i] = (
                np.percentile(boot_rhos, low_pct) if boot_rhos else np.nan
            )
            rho_ci_high.iloc[i, j] = rho_ci_high.iloc[j, i] = (
                np.percentile(boot_rhos, high_pct) if boot_rhos else np.nan
            )

    return {
        "kendall_tau": tau_matrix,
        "kendall_pvalues": tau_pvals,
        "kendall_ci_lower": tau_ci_low,
        "kendall_ci_upper": tau_ci_high,
        "spearman_rho": rho_matrix,
        "spearman_pvalues": rho_pvals,
        "spearman_ci_lower": rho_ci_low,
        "spearman_ci_upper": rho_ci_high,
    }

In [4]:
def compute_polychoric_matrix(
    df: pd.DataFrame, smooth_zeros: bool = True
) -> pd.DataFrame:
    """Computes the Polychoric Correlation Matrix for an ordinal dataset

    (e.g., 27x9 dataframe rated on a discrete 0-2 scale).

    Uses a two-step Maximum Likelihood Estimation (MLE):
    1. Estimates marginal latent thresholds via inverse standard normal CDF.
    2. Optimizes the bivariate latent correlation (rho) using bivariate normal joint CDF.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame of shape (N, P) containing discrete ordinal values (e.g., 0, 1, 2).
    smooth_zeros : bool
        If True, applies a Laplace smoothing (+0.5) to empty cells in small N 3x3 tables
        to prevent boundary crash issues and numerical convergence failure.

    Returns:
    --------
    pd.DataFrame:
        A symmetric (P x P) DataFrame containing the estimated polychoric correlation coefficients.
    """
    cols = df.columns
    n_cols = len(cols)
    poly_matrix = pd.DataFrame(np.eye(n_cols), index=cols, columns=cols)

    def _estimate_bivariate_rho(x: np.ndarray, y: np.ndarray) -> float:
        # Construct contingency table
        categories_x = np.sort(np.unique(x))
        categories_y = np.sort(np.unique(y))

        # Build empirical frequency table
        freq_table = pd.crosstab(
            pd.Categorical(x, categories=categories_x),
            pd.Categorical(y, categories=categories_y),
            dropna=False,
        ).values.astype(float)

        # Apply small-sample zero-cell correction (Laplace-style smoothing)
        if smooth_zeros and np.any(freq_table == 0):
            freq_table += 0.5

        n_total = np.sum(freq_table)
        p_table = freq_table / n_total

        # Step 1: Marginal cumulative proportions and latent normal thresholds
        p_x_cum = np.cumsum(np.sum(p_table, axis=1))[:-1]
        p_y_cum = np.cumsum(np.sum(p_table, axis=0))[:-1]

        # Prevent -inf / inf boundary issues
        p_x_cum = np.clip(p_x_cum, 1e-6, 1 - 1e-6)
        p_y_cum = np.clip(p_y_cum, 1e-6, 1 - 1e-6)

        tau_x = np.concatenate(([-np.inf], norm.ppf(p_x_cum), [np.inf]))
        tau_y = np.concatenate(([-np.inf], norm.ppf(p_y_cum), [np.inf]))

        r_dim, c_dim = freq_table.shape

        # Step 2: Negative Log-Likelihood optimization over latent correlation rho
        def _neg_log_likelihood(rho: float) -> float:
            # Bound rho strictly inside (-1, 1)
            rho = np.clip(rho, -0.999, 0.999)
            cov_mat = np.array([[1.0, rho], [rho, 1.0]])
            mvn = multivariate_normal(mean=[0, 0], cov=cov_mat, allow_singular=True)

            log_lik = 0.0
            for r in range(r_dim):
                for c in range(c_dim):
                    if freq_table[r, c] > 0:
                        # Compute joint probability mass in rectangle [tau_x[r], tau_x[r+1]] x [tau_y[c], tau_y[c+1]]
                        p_cell = (
                            mvn.cdf([tau_x[r + 1], tau_y[c + 1]])
                            - mvn.cdf([tau_x[r], tau_y[c + 1]])
                            - mvn.cdf([tau_x[r + 1], tau_y[c]])
                            + mvn.cdf([tau_x[r], tau_y[c]])
                        )
                        p_cell = max(p_cell, 1e-12)
                        log_lik += freq_table[r, c] * np.log(p_cell)

            return -log_lik

        # Bounded scalar optimization for rho in [-0.999, 0.999]
        res = minimize_scalar(
            _neg_log_likelihood, bounds=(-0.999, 0.999), method="bounded"
        )
        return float(res.x)

    for i in range(n_cols):
        for j in range(i + 1, n_cols):
            x_vals = df[cols[i]].values
            y_vals = df[cols[j]].values

            try:
                rho_est = _estimate_bivariate_rho(x_vals, y_vals)
            except Exception:
                rho_est = np.nan

            poly_matrix.iloc[i, j] = poly_matrix.iloc[j, i] = rho_est

    return poly_matrix

In [5]:
def perform_dimensionality_and_efa(
    df: pd.DataFrame,
    corr_matrix: pd.DataFrame = None,
    rotation: str = "oblimin",
    kaiser_threshold: float = 1.0,
) -> dict:
    """
    Evaluates matrix factorability (KMO & Bartlett's test) and conducts both PCA
    and Exploratory Factor Analysis (EFA) with oblique rotation.

    Parameters:
    -----------
    df : pd.DataFrame
        Original (N x P) DataFrame (e.g., 27x9 consensus ratings).
    corr_matrix : pd.DataFrame, optional
        Precomputed correlation matrix (e.g., Polychoric correlation matrix).
        If None, the standard Pearson/Kendall correlation is calculated from df.
    rotation : str
        Factor rotation technique (default: 'oblimin' for oblique rotation).
    kaiser_threshold : float
        Eigenvalue cutoff criterion for factor retention (default: 1.0).

    Returns:
    --------
    dict:
        Dictionary containing:
        - 'kmo_total': Overall KMO sampling adequacy score
        - 'kmo_per_item': Item-level KMO series
        - 'bartlett_chi_sq': Bartlett's sphericity test statistic
        - 'bartlett_p_val': Bartlett's p-value
        - 'eigenvalues': Array of eigenvalues for all components
        - 'n_factors_retained': Number of components with eigenvalue > 1
        - 'pca_variance_explained': DataFrame of variance breakdown per component
        - 'factor_loadings': DataFrame of rotated pattern matrix (EFA)
        - 'factor_structure': DataFrame of structure matrix (correlations with factors)
        - 'factor_clustering': Dictionary grouping metrics by their primary factor
        - 'cronbach_alphas': Dictionary of internal consistency alpha per retained cluster
    """
    n_rows, n_cols = df.shape
    cols = df.columns

    # Use provided correlation matrix or compute from df
    if corr_matrix is None:
        r_matrix = df.corr()
    else:
        r_matrix = corr_matrix.copy()

    # --- 1. Matrix Factorability: KMO and Bartlett's Test ---
    # Bartlett's test of sphericity
    try:
        chi_square_value, p_value = calculate_bartlett_sphericity(df)
    except Exception:
        # Fallback calculation using correlation determinant
        det_r = np.linalg.det(r_matrix.values)
        det_r = max(det_r, 1e-12)
        chi_sq = - (n_rows - 1 - (2 * n_cols + 5) / 6) * np.log(det_r)
        dof = n_cols * (n_cols - 1) / 2
        p_val = chi2.sf(chi_sq, dof)
        chi_square_value, p_value = chi_sq, p_val

    # Kaiser-Meyer-Olkin (KMO) Measure of Sampling Adequacy
    kmo_per_item, kmo_total = calculate_kmo(df)
    kmo_per_item_series = pd.Series(kmo_per_item, index=cols, name="KMO")

    # --- 2. Initial Eigenvalue Extraction (Kaiser's Criterion) ---
    eigenvalues, _ = np.linalg.eigh(r_matrix.values)
    eigenvalues = np.sort(eigenvalues)[::-1]  # Sort descending

    n_factors = int(np.sum(eigenvalues >= kaiser_threshold))
    n_factors = max(1, min(n_factors, n_cols - 1))  # Safe bound

    # Compute Variance Explained Breakdown
    var_explained = (eigenvalues / n_cols) * 100
    cum_var_explained = np.cumsum(var_explained)
    pca_summary = pd.DataFrame({
        "Eigenvalue": eigenvalues,
        "% Variance": var_explained,
        "Cumulative %": cum_var_explained
    }, index=[f"Component {i+1}" for i in range(n_cols)])

    # --- 3. Exploratory Factor Analysis (EFA) with Oblique Rotation ---
    fa = FactorAnalyzer(
        n_factors=n_factors,
        rotation=rotation,
        method="minres",
        is_corr_matrix=False
    )
    fa.fit(df)

    factor_names = [f"Factor {i+1}" for i in range(n_factors)]
    pattern_loadings = pd.DataFrame(
        fa.loadings_, index=cols, columns=factor_names
    )

    # Structure Matrix (Loadings * Factor Correlation Matrix)
    if fa.phi_ is not None and n_factors > 1:
        structure_loadings = pd.DataFrame(
            np.dot(fa.loadings_, fa.phi_),
            index=cols,
            columns=factor_names
        )
    else:
        structure_loadings = pattern_loadings.copy()

    # --- 4. Metric Clustering & Scale Internal Consistency ---
    # Assign each metric to its highest absolute loading factor
    primary_factor_assignments = pattern_loadings.abs().idxmax(axis=1)
    clustering = {f: [] for f in factor_names}
    for metric, factor in primary_factor_assignments.items():
        clustering[factor].append(metric)

    # Calculate Cronbach's alpha for each identified cluster
    def _calculate_cronbach_alpha(sub_df: pd.DataFrame) -> float:
        k = sub_df.shape[1]
        if k <= 1:
            return np.nan
        item_variances = sub_df.var(axis=0, ddof=1).sum()
        total_score_variance = sub_df.sum(axis=1).var(ddof=1)
        if total_score_variance == 0:
            return 0.0
        return float((k / (k - 1)) * (1 - (item_variances / total_score_variance)))

    alphas = {}
    for factor, metrics in clustering.items():
        if len(metrics) > 1:
            alphas[factor] = _calculate_cronbach_alpha(df[metrics])
        else:
            alphas[factor] = np.nan

    return {
        "kmo_total": float(kmo_total),
        "kmo_per_item": kmo_per_item_series,
        "bartlett_chi_sq": float(chi_square_value),
        "bartlett_p_val": float(p_value),
        "eigenvalues": eigenvalues,
        "n_factors_retained": n_factors,
        "pca_variance_explained": pca_summary,
        "factor_loadings": pattern_loadings,
        "factor_structure": structure_loadings,
        "factor_clustering": clustering,
        "cronbach_alphas": alphas
    }

In [6]:
def compute_ordinal_reliability(
    df: pd.DataFrame,
    poly_corr_matrix: pd.DataFrame = None,
    n_bootstraps: int = 2000,
    alpha_ci: float = 0.05,
) -> dict:
    """Computes Ordinal Cronbach's Alpha and Composite Reliability (McDonald's Omega)

    based on the Polychoric Correlation Matrix for ordinal Likert-scale data.

    Formulas used:
    --------------
    1. Ordinal Cronbach's Alpha:
       alpha_ord = (k / (k - 1)) * (1 - tr(R_poly) / sum(R_poly))
       where k is the number of items and R_poly is the polychoric correlation matrix.

    2. Composite Reliability (McDonald's Omega - 1-factor model):
       omega = (sum(lambda_i))^2 / [ (sum(lambda_i))^2 + sum(theta_i) ]
       where lambda_i are standardized factor loadings and theta_i = 1 - lambda_i^2 are unique variances.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame of shape (N, P) containing discrete ordinal values (e.g., 27x9 on 0-2 scale).
    poly_corr_matrix : pd.DataFrame, optional
        Precomputed (P x P) Polychoric correlation matrix. If None, df.corr() is used.
    n_bootstraps : int
        Number of bootstrap iterations for CI estimation (default: 2000).
    alpha_ci : float
        Significance level for confidence intervals (default: 0.05 for 95% CIs).

    Returns:
    --------
    dict:
        Dictionary containing point estimates, 95% bootstrap CIs, and factor parameters.
    """
    n_rows, k = df.shape
    cols = df.columns

    if poly_corr_matrix is None:
        r_mat = df.corr().values
    else:
        r_mat = poly_corr_matrix.values.copy()

    # --- 1. Ordinal Cronbach's Alpha ---
    def _calc_ordinal_alpha(r: np.ndarray) -> float:
        p = r.shape[0]
        if p <= 1:
            return np.nan
        sum_total = np.sum(r)
        trace_r = np.trace(r)  # Equal to p for correlation matrices
        if sum_total == 0:
            return 0.0
        return float((p / (p - 1)) * (1.0 - (trace_r / sum_total)))

    ordinal_alpha = _calc_ordinal_alpha(r_mat)

    # Standard (Pearson-based) Cronbach's Alpha for comparison
    def _calc_standard_alpha(data_matrix: np.ndarray) -> float:
        p = data_matrix.shape[1]
        item_vars = np.var(data_matrix, axis=0, ddof=1).sum()
        total_var = np.var(data_matrix.sum(axis=1), ddof=1)
        if total_var == 0:
            return 0.0
        return float((p / (p - 1)) * (1.0 - (item_vars / total_var)))

    standard_alpha = _calc_standard_alpha(df.values)

    # --- 2. Composite Reliability / McDonald's Omega (1-Factor CFA Model) ---
    def _calc_omega_from_corr(r: np.ndarray) -> tuple:
        fa = FactorAnalyzer(
            n_factors=1, rotation=None, method="minres", is_corr_matrix=True
        )
        fa.fit(r)
        loadings = fa.loadings_.flatten()
        # Bound loadings to [-1, 1]
        loadings = np.clip(loadings, -0.999, 0.999)
        uniquenesses = 1.0 - loadings**2

        sum_loadings = np.sum(loadings)
        sum_uniqueness = np.sum(uniquenesses)

        omega = (sum_loadings**2) / (sum_loadings**2 + sum_uniqueness)
        return float(omega), loadings, uniquenesses

    omega_val, lambda_loadings, unique_vars = _calc_omega_from_corr(r_mat)

    # --- 3. Small-Sample Bootstrapping for 95% Confidence Intervals (N = 27) ---
    boot_alphas = []
    boot_omegas = []

    np.random.seed(42)
    for _ in range(n_bootstraps):
        boot_idx = np.random.choice(n_rows, size=n_rows, replace=True)
        boot_sample = df.iloc[boot_idx].values

        # Estimate correlation matrix on bootstrap resample
        boot_corr = np.corrcoef(boot_sample, rowvar=False)

        # Catch potential collinearity in small bootstrap resamples
        if not np.any(np.isnan(boot_corr)):
            b_alpha = _calc_ordinal_alpha(boot_corr)
            try:
                b_omega, _, _ = _calc_omega_from_corr(boot_corr)
            except Exception:
                b_omega = np.nan

            if not np.isnan(b_alpha):
                boot_alphas.append(b_alpha)
            if not np.isnan(b_omega):
                boot_omegas.append(b_omega)

    low_pct = (alpha_ci / 2) * 100
    high_pct = (1 - alpha_ci / 2) * 100

    alpha_ci_bounds = (
        (np.percentile(boot_alphas, low_pct), np.percentile(boot_alphas, high_pct))
        if boot_alphas
        else (np.nan, np.nan)
    )
    omega_ci_bounds = (
        (np.percentile(boot_omegas, low_pct), np.percentile(boot_omegas, high_pct))
        if boot_omegas
        else (np.nan, np.nan)
    )

    loadings_summary = pd.DataFrame(
        {
            "Standardized_Loading (λ)": lambda_loadings,
            "Error_Variance (θ)": unique_vars,
        },
        index=cols,
    )

    return {
        "ordinal_cronbach_alpha": ordinal_alpha,
        "ordinal_alpha_ci_95": alpha_ci_bounds,
        "composite_reliability_omega": omega_val,
        "omega_ci_95": omega_ci_bounds,
        "standard_cronbach_alpha": standard_alpha,
        "single_factor_loadings": loadings_summary,
    }

In [7]:
def compare_metric_groups(
    df: pd.DataFrame, grouping_series: pd.Series, group_name: str = "Group"
) -> dict:
    """Performs non-parametric group comparisons (Wilcoxon Rank-Sum / Mann-Whitney U

    for 2 groups, Kruskal-Wallis H-test for >= 3 groups) across all 9 metrics.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame of shape (N, P) containing metric ratings (e.g., 27x9 on 0-2 scale).
    grouping_series : pd.Series
        Categorical grouping variable for the N items (e.g., paper type: Qualitative vs. Quantitative).
    group_name : str
        Label for the grouping variable.

    Returns:
    --------
    dict:
        Dictionary containing:
        - 'descriptive_stats': Median, IQR, Mean, and SD per metric by group.
        - 'overall_tests': DataFrame of Wilcoxon (z, p, effect size r) or Kruskal-Wallis (H, p, eta_sq).
        - 'pairwise_comparisons': Post-hoc pairwise Wilcoxon tests with Bonferroni-adjusted p-values (if >= 3 groups).
    """
    metrics = df.columns
    combined_df = df.copy()
    combined_df[group_name] = grouping_series.values

    # Remove any rows with missing group labels
    combined_df = combined_df.dropna(subset=[group_name])
    unique_groups = combined_df[group_name].unique()
    k_groups = len(unique_groups)

    if k_groups < 2:
        raise ValueError(
            f"Need at least 2 distinct groups to compare, found: {unique_groups}"
        )

    # --- 1. Descriptive Statistics per Group ---
    desc_list = []
    for grp in unique_groups:
        grp_data = combined_df[combined_df[group_name] == grp][metrics]
        medians = grp_data.median()
        iqrs = grp_data.quantile(0.75) - grp_data.quantile(0.25)
        means = grp_data.mean()
        sds = grp_data.std()
        counts = grp_data.count()

        grp_summary = pd.DataFrame(
            {
                "Group": grp,
                "N": counts,
                "Median": medians,
                "IQR": iqrs,
                "Mean": means,
                "SD": sds,
            }
        )
        desc_list.append(grp_summary)

    desc_df = (
        pd.concat(desc_list)
        .reset_index()
        .rename(columns={"index": "Metric"})
        .sort_values(by=["Metric", "Group"])
    )

    # --- 2. Statistical Significance Testing ---
    results = []
    pairwise_results = []

    for metric in metrics:
        group_samples = [
            combined_df[combined_df[group_name] == grp][metric].dropna().values
            for grp in unique_groups
        ]

        if k_groups == 2:
            # Two groups: Wilcoxon Rank-Sum Test
            g1, g2 = group_samples[0], group_samples[1]
            stat, p_val = ranksums(g1, g2)
            n_total = len(g1) + len(g2)
            # Effect size r = |z| / sqrt(N)
            effect_r = abs(stat) / np.sqrt(n_total) if n_total > 0 else np.nan

            results.append(
                {
                    "Metric": metric,
                    "Test": "Wilcoxon Rank-Sum",
                    "Statistic (z)": stat,
                    "p_value": p_val,
                    "Effect_Size (r)": effect_r,
                    "Significant (p < .05)": p_val < 0.05,
                }
            )

        else:
            # 3 or more groups: Kruskal-Wallis H-test
            stat, p_val = kruskal(*group_samples)
            n_total = sum(len(g) for g in group_samples)
            # Effect size: Epsilon-squared / eta-squared = (H - k + 1) / (N - k)
            eta_sq = (
                (stat - k_groups + 1) / (n_total - k_groups)
                if (n_total - k_groups) > 0
                else np.nan
            )
            eta_sq = max(0.0, eta_sq)  # Bound at 0

            results.append(
                {
                    "Metric": metric,
                    "Test": "Kruskal-Wallis",
                    "Statistic (H)": stat,
                    "p_value": p_val,
                    "Effect_Size (eta_sq)": eta_sq,
                    "Significant (p < .05)": p_val < 0.05,
                }
            )

            # Post-hoc pairwise Wilcoxon tests with Bonferroni correction
            pairs = list(combinations(range(k_groups), 2))
            n_comparisons = len(pairs)

            for i, j in pairs:
                p_g1, p_g2 = group_samples[i], group_samples[j]
                p_stat, p_raw_pval = ranksums(p_g1, p_g2)
                p_adj_pval = min(1.0, p_raw_pval * n_comparisons)
                pair_n = len(p_g1) + len(p_g2)
                pair_r = (
                    abs(p_stat) / np.sqrt(pair_n) if pair_n > 0 else np.nan
                )

                pairwise_results.append(
                    {
                        "Metric": metric,
                        "Comparison": f"{unique_groups[i]} vs {unique_groups[j]}",
                        "Statistic (z)": p_stat,
                        "p_raw": p_raw_pval,
                        "p_bonferroni": p_adj_pval,
                        "Effect_Size (r)": pair_r,
                        "Significant": p_adj_pval < 0.05,
                    }
                )

    test_df = pd.DataFrame(results)
    pairwise_df = pd.DataFrame(pairwise_results) if pairwise_results else None

    return {
        "descriptive_stats": desc_df,
        "overall_tests": test_df,
        "pairwise_comparisons": pairwise_df,
    }

In [8]:
def compute_metric_distribution_summary(
    df: pd.DataFrame,
    scale_values: list = [0, 1, 2],
    include_parametric: bool = True,
) -> pd.DataFrame:
    """Computes frequency distributions, proportions, medians, IQRs, and

    confidence intervals for discrete 3-point Likert scale metrics (0, 1, 2).

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame of shape (N, P) containing metric ratings (e.g., 27x9).
    scale_values : list
        The complete set of possible categorical scale values (default: [0, 1, 2]).
    include_parametric : bool
        Whether to append secondary Mean and SD for reporting completeness.

    Returns:
    --------
    pd.DataFrame:
        A structured summary table containing counts, percentages, and ordinal
        descriptives for each metric.
    """
    n_rows = len(df)
    summary_rows = []

    for col in df.columns:
        series = df[col].dropna()
        n_valid = len(series)

        # 1. Frequency counts and proportions across the 0-2 scale
        val_counts = series.value_counts()
        freq_dict = {}
        for val in scale_values:
            count = val_counts.get(val, 0)
            pct = (count / n_valid) * 100 if n_valid > 0 else 0.0
            freq_dict[f"Count_{val}"] = int(count)
            freq_dict[f"Pct_{val}"] = np.round(pct, 1)

        # 2. Ordinal / Non-parametric Descriptives
        median_val = series.median()
        q25 = series.quantile(0.25)
        q75 = series.quantile(0.75)
        iqr_val = q75 - q25

        row = {
            "Metric": col,
            "N": n_valid,
            **freq_dict,
            "Median": median_val,
            "Q1 (25%)": q25,
            "Q3 (75%)": q75,
            "IQR": iqr_val,
        }

        # 3. Optional Parametric Comparison Values
        if include_parametric:
            row["Mean"] = np.round(series.mean(), 2)
            row["SD"] = np.round(series.std(ddof=1), 2)

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows).set_index("Metric")
    return summary_df

In [12]:
df = pd.read_excel("/home/david/Git/SDLT_Paper/Data/adiya_isla_scores.xlsx", sheet_name="Whole")  # Replace with your actual data path and sheet name
grouping_series = None
group_name = "Publication"
export_to_excel = True
excel_filename = "/home/david/Git/SDLT_Paper/Data/metric_evaluation_report.xlsx"
results = {}

In [13]:
# --- Step 1: Metric Distributions, Medians, and IQRs ---
print("[1/5] Computing frequency distributions, medians, and IQRs...")
dist_summary = compute_metric_distribution_summary(df, scale_values=[0, 1, 2])
results["distribution_summary"] = dist_summary

[1/5] Computing frequency distributions, medians, and IQRs...


TypeError: Cannot perform reduction 'median' with string dtype

In [14]:
# --- Step 2: Ordinal Correlation Matrices (Kendall's tau-b & Spearman's rho) ---
print("[2/5] Calculating Kendall's tau-b and Spearman's rho with bootstrap CIs...")
ordinal_corr_results = compute_ordinal_correlations(
    df, n_bootstraps=2000, alpha=0.05
)
results["ordinal_correlations"] = ordinal_corr_results

[2/5] Calculating Kendall's tau-b and Spearman's rho with bootstrap CIs...


AttributeError: 'float' object has no attribute 'dtype'

In [15]:
# --- Step 3: Latent Polychoric Correlation Matrix ---
print("[3/5] Estimating 9x9 Polychoric Correlation Matrix via Two-Step MLE...")
poly_matrix = compute_polychoric_matrix(df, smooth_zeros=True)
results["polychoric_matrix"] = poly_matrix

[3/5] Estimating 9x9 Polychoric Correlation Matrix via Two-Step MLE...


In [16]:
# --- Step 4: Factorability, PCA, and EFA Clustering ---
print("[4/5] Evaluating matrix factorability (KMO/Bartlett) and EFA clustering...")
efa_results = perform_dimensionality_and_efa(
    df=df,
    corr_matrix=poly_matrix,
    rotation="oblimin",
    kaiser_threshold=1.0,
)
results["factor_analysis"] = efa_results

[4/5] Evaluating matrix factorability (KMO/Bartlett) and EFA clustering...


/home/david/Git/SDLT_Paper/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2406: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


TypeError: can only concatenate str (not "float") to str

In [17]:
# --- Step 5: Ordinal Reliability (Alpha & Omega) ---
print("[5/5] Calculating Ordinal Cronbach's Alpha and Composite Reliability (Omega)...")
reliability_results = compute_ordinal_reliability(
    df=df,
    poly_corr_matrix=poly_matrix,
    n_bootstraps=2000,
    alpha_ci=0.05,
)
results["reliability"] = reliability_results

[5/5] Calculating Ordinal Cronbach's Alpha and Composite Reliability (Omega)...


TypeError: can only concatenate str (not "float") to str

In [18]:

# --- Step 6: Optional Non-Parametric Group Comparisons ---
if grouping_series is not None:
    print(f"[*] Conducting group comparisons across '{group_name}'...")
    group_comp_results = compare_metric_groups(
        df=df, grouping_series=grouping_series, group_name=group_name
    )
    results["group_comparisons"] = group_comp_results

In [ ]:

# --- Step 7: Export Structured Results to Excel ---
if export_to_excel:
    print(f"[*] Exporting all summary tables to '{excel_filename}'...")
    with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
        # Sheet 1: Descriptives
        dist_summary.to_excel(writer, sheet_name="Descriptives")

        # Sheet 2 & 3: Ordinal Correlations
        ordinal_corr_results["kendall_tau"].to_excel(writer, sheet_name="Kendall_Tau")
        ordinal_corr_results["spearman_rho"].to_excel(writer, sheet_name="Spearman_Rho")

        # Sheet 4: Polychoric Correlation Matrix
        poly_matrix.to_excel(writer, sheet_name="Polychoric_Matrix")

        # Sheet 5: EFA Pattern Matrix & Variance
        efa_results["factor_loadings"].to_excel(writer, sheet_name="EFA_Loadings")
        efa_results["pca_variance_explained"].to_excel(writer, sheet_name="PCA_Variance")

        # Sheet 6: Reliability Loadings
        reliability_results["single_factor_loadings"].to_excel(writer, sheet_name="Reliability_Loadings")

        # Sheet 7: Group Comparisons (if provided)
        if grouping_series is not None:
            results["group_comparisons"]["overall_tests"].to_excel(
                writer, sheet_name="Group_Tests", index=False
            )